In [161]:
import pandas as pd
import numpy as np
import uuid
import random

from faker import Faker
from dateutil.relativedelta import relativedelta

In [162]:
objects_df = pd.read_csv('../data/objects.csv')
constituents_df = pd.read_csv('../data/constituents.csv')
links_df = pd.read_csv('../data/objects_constituents.csv')
text_df = pd.read_csv('../data/objects_text_entries.csv')

# artist_df = pd.read_csv('../data/constituents.csv')
# artwork_df = pd.read_csv('../data/objects.csv')
# customer_df = pd.read_csv('../data/constituents.csv')
# exhibition_df = pd.read_csv('../data/objects_text_entries.csv')
# category_df = pd.read_csv('../data/objects.csv')


/var/folders/kt/3q9gkxcj2jx29zmt127wqyhm0000gn/T/ipykernel_85640/2375069269.py:1: DtypeWarning: Columns (24) have mixed types. Specify dtype option on import or set low_memory=False.
  objects_df = pd.read_csv('../data/objects.csv')
/var/folders/kt/3q9gkxcj2jx29zmt127wqyhm0000gn/T/ipykernel_85640/2375069269.py:3: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  links_df = pd.read_csv('../data/objects_constituents.csv')


In [163]:
# Helper functions
def generate_fake_location(locale='en_US'):
    """Generates a single fake address, city, state, and datetime."""
    fake = Faker(locale)
    
    return {
        "address": fake.street_address(),
        "city": fake.city(),
        "state": fake.state_abbr(),
        "timestamp": fake.date_time_this_year()
    }


def generate_fake_date_range(min_months=1, max_months=6):
    """
    Generates a start and end date separated by a random number of months.
    Returns: (start_date, end_date) as datetime objects.
    """
    fake = Faker()
    
    # 1. Generate a random start date (e.g., within the last 2 years)
    start_date = fake.date_time_between(start_date='-2y', end_date='now')
    
    # 2. Pick a random number of months for the duration
    duration_months = random.randint(min_months, max_months)
    
    # 3. Calculate end date by adding the months
    end_date = start_date + relativedelta(months=duration_months)
    
    return start_date, end_date


def generate_id():
    """Generates a random UUID4 string."""
    return str(uuid.uuid4())

def generate_numeric_id():
    """Generates a unique ID based on the current timestamp in nanoseconds."""
    return random.randint(10_000_000, 99_999_999)


def generate_art_theme():
    """Generates a random string representing an art exhibition theme."""
    
    # Artistic movements or styles
    themes = [
        "Florals", "Black Culture", "Women in Art", "Landscapes", 
        "Portraits", "Urban Life", "Maritime History", "Abstract Forms",
        "The Human Condition", "Mythology", "Indigenous Art", "Architecture",
        "17th Century French", "Impressionism", "Modernism", "Baroque", 
        "Surrealism", "Renaissance", "Contemporary", "Post-Modern", 
        "Fauvism", "Expressionism", "Minimalism", "Romanticism"
    ]
    
    return f"{random.choice(themes)}"


def get_availability():
    """Returns a random string: either 'available' or 'unavailable'."""
    options = ["available", "unavailable"]
    return random.choice(options)

In [164]:
# Artist
artist_df = constituents_df[constituents_df['artistofngaobject'] == 1].copy()
artist_df = constituents_df[['constituentid', 'preferreddisplayname']]
artist_df = pd.concat([artist_df, objects_df[['displaydate', 'visualbrowsertimespan', 'visualbrowserclassification']]], axis=1)
artist_df.columns = ['artistID', 'artistName', 'birthplace', 'period', 'type']
artist_df = artist_df.dropna(subset=['artistID', 'artistName']) # keep row if both exist

# generate numeric id's
for idx, row in artist_df.iterrows():
    artist_df.at[idx, 'artistID'] = generate_numeric_id()
artist_df['artistID'] = artist_df['artistID'].round().convert_dtypes() # convert to ints




# Customers 
#  use the link table to find people who gave art to the NGA
donor_ids = links_df[links_df['roletype'] == 'donor']['constituentid'].unique()
customer_df = constituents_df[constituents_df['constituentid'].isin(donor_ids)].copy()
customer_df = customer_df[['constituentid', 'forwarddisplayname']].dropna()
customer_df.columns = ['customerID', 'customerName']
customer_df['address'] = None
customer_df['donationAmount'] = None
customer_df['donationDate'] = None

# generate address and donations
for idx, row in customer_df.iterrows():
    address_dict = generate_fake_location()
    customer_df.at[idx, 'address'] = address_dict['address']
    customer_df['donationAmount'] = np.random.uniform(500, 50000, size=len(customer_df)).round(2)
    customer_df.at[idx, 'donationDate'] = pd.to_datetime(address_dict['timestamp'])





# Artwork - still need price
artwork_df = objects_df[objects_df['isvirtual'] == 0].copy()
artwork_df = artwork_df[['objectid', 'title', 'displaydate', 'visualbrowserclassification', 'medium']]
artwork_df.columns = ['artworkID', 'title', 'creationYear', 'type', 'medium']
artwork_df = artwork_df.dropna()


artwork_df['price'] = np.random.uniform(1000, 1000000, size=len(artwork_df)).round(2)

# map artistID from the link table
artist_map = links_df[links_df['roletype'] == 'artist'].groupby('objectid')['constituentid'].first()
artwork_df['artistID'] = artwork_df['artworkID'].map(artist_map)

artwork_df['canPurchase'] = [get_availability() for _ in range(len(artwork_df))] # available to purchase

# map random customerID to simulate a purchase

# if artwork is available to purchase, no customerID, else if unavailable, its either bought or not available for purchase
options = [random.choice(customer_df['customerID'].values), None]
artwork_df['customerID'] = artwork_df['canPurchase'].apply(
    lambda x: None if x == "available" else np.random.choice(options)
)

artwork_df = artwork_df.dropna(subset=['artistID']) # ensure every piece has an artist




# Categories - create unique category list based on NGA classifications
category_df = objects_df[['classification', 'visualbrowsertimespan', 'subclassification']].drop_duplicates()

# generate an id
category_df['classificationid'] = None
for idx, row in category_df.iterrows():
    category_df.at[idx, 'classificationid'] = generate_id()

# move id column to front
move_column = 'classificationid'
category_df.insert(0, 'classificationid', category_df.pop('classificationid'))
category_df = category_df.dropna()

# rename columns
category_df.columns = ['categoryID', 'categoryName', 'period', 'type']
category_df = category_df.dropna(subset=['categoryID'])




# Exhibitions - filter text entries for exhibition history
ex_data = text_df[text_df['texttype'] == 'exhibition_history'].copy()

# simplify and treat each unique year/text combo as an event
exhibition_df = ex_data[['text', 'year']].drop_duplicates().head(500) 
exhibition_df['exhibitionID'] = range(1, len(exhibition_df) + 1)
exhibition_df.columns = ['exhibitionTitle', 'theme', 'exhibitionID']
exhibition_df['startDate'] = None
exhibition_df['endDate'] = None

# generate random start and end dates for exhibition
for idx, row in exhibition_df.iterrows():
    start, end = generate_fake_date_range()
    exhibition_df.at[idx, 'startDate'] = start
    exhibition_df.at[idx, 'endDate'] = end
    exhibition_df.at[idx, 'theme'] = generate_art_theme()





# Relationship Tables 

# Displays (Artwork <-> Exhibition)
displays_df = artwork_df[['artistID']].copy()
# Assign random exhibition IDs for the sake of the project join requirements
displays_df['exhibitionID'] = np.random.choice(exhibition_df['exhibitionID'], size=len(displays_df))
displays_df.columns = ['artworkID', 'exhibitionID']
displays_df = displays_df.dropna(subset=['artworkID', 'exhibitionID'])



# Belongs (Artwork <-> Category)
belongs_df = pd.DataFrame()
belongs_df = pd.concat([belongs_df, artist_df[['artistID']]], axis=1)
belongs_df = pd.concat([belongs_df, category_df[['categoryID']]], axis=1)
belongs_df = belongs_df.dropna(subset=['artistID', 'categoryID'])



# Attends (Customer <-> Exhibition)
attends_df = pd.DataFrame()
attends_df = pd.concat([attends_df, customer_df[['customerID']]], axis=1)
attends_df['exhibitionID'] = np.random.choice(exhibition_df['exhibitionID'], size=len(attends_df))
attends_df = attends_df.dropna(subset=['customerID', 'exhibitionID'])

/var/folders/kt/3q9gkxcj2jx29zmt127wqyhm0000gn/T/ipykernel_85640/3668884644.py:100: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Contemporary' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  exhibition_df.at[idx, 'theme'] = generate_art_theme()


In [166]:
artist_df.to_csv('../data/artists.csv', index=False)
artwork_df.to_csv('../data/artwork.csv', index=False)
customer_df.to_csv('../data/customers.csv', index=False)
exhibition_df.to_csv('../data/exhibitions.csv', index=False)
category_df.to_csv('../data/categories.csv', index=False)
displays_df.to_csv('../data/displays.csv', index=False)
belongs_df.to_csv('../data/belongs.csv', index=False)
attends_df.to_csv('../data/attends.csv', index=False)